We said in our second notebook that we would have compared the pre-tokenized data kindly given to us by the CodeXGLUE database, with a BPE tokenizer. That is because, as explained in this [paper](https://www.researchgate.net/publication/394590942_How_Different_Tokenization_Algorithms_Impact_LLMs_and_Transformer_Models_for_Binary_Code_Analysis) published in 2025, that seems to be the best combination with an LSTM for code summarization. 

We are gonna try to use the [Huggin Face tokenizers](https://github.com/huggingface/tokenizers).

In [1]:
import ast

def is_ast_valid(example):
	try:
		ast.parse(example['code'])
		return True
	except SyntaxError:
		return False

In [2]:
from datasets import load_dataset
train_dataset = load_dataset(
	"json", 
	data_files="./../../data/raw/dataset/python/train.jsonl",
	split="train"
).filter(is_ast_valid)
valid_dataset = load_dataset(
	"json",
	data_files="./../../data/raw/dataset/python/valid.jsonl",
	split="train"
).filter(is_ast_valid)
test_dataset = load_dataset(
	"json",
	data_files="./../../data/raw/dataset/python/test.jsonl",
	split="train"
).filter(is_ast_valid)

c:\Users\Sean Andreini\Desktop\Unifi\Machine Learning for Software Analysis\code-summarization-mlsa-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Now, if you remember there's a little difference between 'docstring_tokens' and 'docstring'. Let's see it:

In [3]:
print(train_dataset['code'][1000])
print(train_dataset['docstring_tokens'][1000])
print(train_dataset['docstring'][1000])

def pair_distance_centile(X, centile, max_pairs=5000):
    """
    Calculate centiles of distances between random pairs in a dataset.

    This an alternative to the median kNN distance for setting the kernel
    length scale.
    """
    N = X.shape[0]
    n_pairs = min(max_pairs, N**2)
    # randorder1 = np.random.permutation(N)
    # randorder2 = np.random.permutation(N)

    dists = np.zeros(n_pairs)

    for i in range(n_pairs):
        pair = np.random.randint(0, N, 2)
        pairdiff = X[pair[0], :]-X[pair[1], :]
        dists[i] = np.dot(pairdiff, pairdiff.T)
    dists.sort()

    out = dists[int(n_pairs*centile/100.)]
    return np.sqrt(out)
['Calculate', 'centiles', 'of', 'distances', 'between', 'random', 'pairs', 'in', 'a', 'dataset', '.']
Calculate centiles of distances between random pairs in a dataset.

    This an alternative to the median kNN distance for setting the kernel
    length scale.


We can see that 'docstring' is much more complex, as it explaines every parameter. We don't want that for 2 main reasons: first of all, we trained our last models with 'docstring_tokens', so it would be unfair to train this differently. Secondly, our vocab would explode, probably tanking our performances. So we're gonna use 'docstring_tokens', as it's just the simple docstring splitted on spaces, and rejoin it, so we can feed it to the tokenizer as if it was the original.

In [4]:
print(" ".join(train_dataset['docstring_tokens'][1]))

Check to make sure the supplied directory path does not exist if so create it . The method catches OSError exceptions and returns a descriptive message instead of re - raising the error .


In [5]:
import os
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import NFD, Lowercase, StripAccents, Sequence

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.normalizer = Sequence([NFD(), Lowercase(), StripAccents()])

tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
	vocab_size = 8000,
	min_frequency = 2,
	special_tokens = ["[UNK]", "[EOS]", "[PAD]", "[BOS]"]
)

tokenizer.train_from_iterator([" ".join(tokens) for tokens in train_dataset['docstring_tokens']], trainer=trainer)

save_path = "./../../data/processed/notebooks/ast_BPE/bpe_tokenizer.json"
os.makedirs(os.path.dirname(save_path), exist_ok=True)

tokenizer.save(save_path)

let's now tokenize the data with our tokenizer and save it.

In [ ]:
train_encoded = [tokenizer.encode(" ".join(docstring)).ids for docstring in train_dataset['docstring_tokens']]
valid_encoded = [tokenizer.encode(" ".join(docstring)).ids for docstring in valid_dataset['docstring_tokens']]
test_encoded = [tokenizer.encode(" ".join(docstring)).ids for docstring in test_dataset['docstring_tokens']]

In [7]:
print(train_encoded[0])

[97, 992, 65, 1021, 69, 4707, 435, 86, 32, 48, 40, 923, 10, 1333, 7129, 203, 11]


In [8]:
from datasets import Dataset, DatasetDict
from datasets import load_from_disk
old_dataset = load_from_disk("../../data/processed/notebooks/ast/")

processed_datasets = DatasetDict({
	'train': Dataset.from_dict({
		'input_ids': old_dataset['train']['input_ids'],
		'labels': train_encoded
	}),
	'valid': Dataset.from_dict({
		'input_ids': old_dataset['valid']['input_ids'],
		'labels': valid_encoded
	}),
	'test': Dataset.from_dict({
		'input_ids': old_dataset['test']['input_ids'],
		'labels': test_encoded
	})
})

In [9]:
from gensim import corpora

ast_dictionary = corpora.Dictionary.load('./../../data/processed/notebooks/ast/code_dictionary.pt')
processed_datasets.save_to_disk('./../../data/processed/notebooks/ast_BPE/')
ast_dictionary.save('./../../data/processed/notebooks/ast_BPE/code_dictionary.pt')

Saving the dataset (1/1 shards): 100%|██████████| 14761/14761 [00:00<00:00, 673301.81 examples/s]
